# Metadata for the 114 unanimous-inclusion cases

Run all cells with Python 3 from this MVP folder or any directory inside the repository.
Only the standard library is required; IPython enables the inline table display.
Outputs are saved in `MVP/data/metadata_v1/`. Original sources, submissions, and the
separate 169-case eligibility audit remain unchanged.

**Target policy:** retain each reviewer's binary label and review status. A unanimous
binary label becomes `provisional_target`; `target` stays null pending Hayden's review.
No Hayden-reviewed decision input is supplied to this notebook. All three current
submissions are `llm_draft`. Agreement does not establish human-reviewed targets.
The codebook defines 0 = fully affirmed; 1 = fully reversed or vacated.

**Family policy:** `case_family_id` stays null pending review. Carry existing family
hints and the explicitly named Coreano/Sodhi warnings as candidate links, including
cases outside the subset. No duplicate investigation or family adjudication occurs;
a case without a warning is not a confirmed singleton.

**Calibration policy:** flag matches in the calibration manifest and saved independent
calibration notes as training-only. No match means no recorded match, not proof of
no prior exposure. Additional previously discussed cases require confirmation.

`decision_date` copies source `date_filed`. Missing separate text-source URLs are
recorded explicitly; every case must still have a source URL and matching text hash.
This metadata is for audit and supervision; label and eligibility fields must not
become ordinary model features. No splits, cleaning, masking, or training are performed.


In [1]:
from pathlib import Path
from collections import Counter
from datetime import date
import hashlib, html, json, re, subprocess

REVIEWERS = ("becca", "hyunsang", "yinkui")
VERSION = "metadata_v1"
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "annotation/source/cases.jsonl").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside the legal-judgment repository.")
MVP = ROOT / "annotation/submissions/becca/MVP"
SELECTION = MVP / "data/unanimous_include_v1"
OUTPUT = MVP / "data" / VERSION
INPUTS = {
    "case_ids": SELECTION / "case_ids.json",
    "cases": SELECTION / "cases.jsonl",
    "selection_manifest": SELECTION / "selection_manifest.json",
    "audit": SELECTION / "eligibility_audit.jsonl",
    "all_sources": ROOT / "annotation/source/cases.jsonl",
    "families": ROOT / "annotation/source/family_hints.json",
    "calibration": ROOT / "annotation/calibration/manifest.json",
    "codebook": ROOT / "annotation/docs/CODEBOOK.md",
    **{n: ROOT / f"annotation/submissions/{n}/batches/A/annotations.jsonl" for n in REVIEWERS},
}
for name in REVIEWERS:
    p = ROOT / f"annotation/submissions/{name}/independent_calibration_notes.md"
    if p.is_file():
        INPUTS[f"{name}_calibration_notes"] = p

def require(condition, message):
    if not condition:
        raise ValueError(message)

def read_json(p):
    return json.loads(p.read_text(encoding="utf-8"))

def read_rows(p):
    return [json.loads(s) for s in p.read_text(encoding="utf-8").splitlines() if s.strip()]

def index(rows):
    result = {r["case_id"]: r for r in rows}
    require(len(result) == len(rows), "Duplicate case IDs")
    return result

def digest(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

ids = read_json(INPUTS["case_ids"])
require(len(ids) == len(set(ids)) == 114, "Expected 114 unique IDs")
sources = index(read_rows(INPUTS["cases"]))
all_sources = index(read_rows(INPUTS["all_sources"]))
audit = index(read_rows(INPUTS["audit"]))
reviews = {n: index(read_rows(INPUTS[n])) for n in REVIEWERS}
selection_manifest = read_json(INPUTS["selection_manifest"])
require(set(sources) == set(ids) and len(audit) == 169, "Unexpected selection/audit coverage")
for name in REVIEWERS:
    require(set(reviews[name]) == set(audit), f"Reviewer coverage mismatch: {name}")
    require(digest(INPUTS[name]) == selection_manifest["inputs"][name]["sha256"],
            f"Reviewer inputs changed: {name}; reconcile the selection version first.")
for filename in ("case_ids.json", "cases.jsonl", "eligibility_audit.jsonl"):
    require(digest(SELECTION / filename) == selection_manifest["outputs"][filename]["sha256"],
            f"Selection file changed: {filename}")
print("Verified 114 fixed case IDs and the separate 169-case audit.")


Verified 114 fixed case IDs and the separate 169-case audit.


## Carry forward documented calibration and family warnings

Candidate family links use existing hints and source case-name matches for the two
explicitly flagged surnames. These are warnings for later review, not resolved families.


In [2]:
calibration_evidence = {}
for cid in read_json(INPUTS["calibration"])["case_ids"]:
    calibration_evidence.setdefault(cid, []).append(str(INPUTS["calibration"].relative_to(ROOT)))
for name, p in INPUTS.items():
    if name.endswith("_calibration_notes"):
        for cid in set(re.findall(r"NY-\d+", p.read_text(encoding="utf-8"))):
            calibration_evidence.setdefault(cid, []).append(str(p.relative_to(ROOT)))
related = {cid: set() for cid in ids}
warnings = {cid: [] for cid in ids}
for group in read_json(INPUTS["families"]).get("groups", []):
    members = set(group["case_ids"])
    for cid in members & set(ids):
        related[cid].update(members - {cid})
        warnings[cid].append(f"Existing hint: {group['basis']} ({group['key']})")
for surname in ("Coreano", "Sodhi"):
    members = {cid for cid, r in all_sources.items()
               if re.search(r"\b" + surname + r"\b", r["case_name"], re.IGNORECASE)}
    for cid in members & set(ids):
        related[cid].update(members - {cid})
        warnings[cid].append(f"Project instruction: review {surname} family warning")
print("Recorded calibration matches:", len(set(ids) & set(calibration_evidence)))
print("Selected cases with family warnings:", sum(bool(v) for v in warnings.values()))


Recorded calibration matches: 3
Selected cases with family warnings: 2


## Build the metadata table

Final target and family fields remain unset. Reviewer labels and unanimous labels
are retained as provisional audit information. `source_word_count` is copied from
the raw source, not calculated for a cleaned FULL/MASKED condition.


In [3]:
metadata = []
for cid in ids:
    src = sources[cid]
    require(src == all_sources[cid], f"Source record changed: {cid}")
    require(bool(src.get("source_url")), f"Missing source URL: {cid}")
    require(isinstance(src.get("text"), str) and bool(src["text"].strip()), f"Empty text: {cid}")
    actual_hash = hashlib.sha256(src["text"].encode("utf-8")).hexdigest()
    require(actual_hash == src["text_sha256"], f"Source hash mismatch: {cid}")
    parsed_date = date.fromisoformat(src["date_filed"])
    labels = [reviews[n][cid]["outcome"]["binary_label"] for n in REVIEWERS]
    require(all(v is None or (type(v) is int and v in (0, 1)) for v in labels), f"Invalid label: {cid}")
    for name in REVIEWERS:
        r = reviews[name][cid]
        require(r["eligibility"]["binary_task"] == "include", f"Not unanimous include: {cid}")
        require(r["text_sha256"] == actual_hash, f"Reviewer/source mismatch: {cid}")
        require(r["run"]["review_status"] == "llm_draft",
                f"Review status changed for {name}/{cid}; update target policy before export.")
    agree = labels[0] is not None and len(set(labels)) == 1
    row = {
        "case_id": cid, "case_name": src["case_name"],
        "source_url": src["source_url"], "text_source_url": src.get("text_source_url") or None,
        "text_source_url_missing": not bool(src.get("text_source_url")),
        "text_sha256": actual_hash,
        "source_record_path": str(INPUTS["all_sources"].relative_to(ROOT)),
        "source_subset_path": str(INPUTS["cases"].relative_to(ROOT)),
        "decision_date": src["date_filed"], "decision_year": parsed_date.year,
        "court": src["court"], "docket_number": src.get("docket_number"),
        "citations": src.get("citations", []), "cluster_id": src.get("cluster_id"),
        "opinion_id": src.get("opinion_id"),
        "eligibility": "include", "eligibility_basis": "unanimous_three_reviewer_include",
        "eligibility_review_status": "llm_draft_consensus_pending_human_review",
        "case_family_id": None,
        "family_review_status": "warning_pending_review" if warnings[cid] else "not_reviewed",
        "candidate_related_case_ids": sorted(related[cid]),
        "family_warnings": list(dict.fromkeys(warnings[cid])),
        "target": None, "provisional_target": labels[0] if agree else None,
        "target_agreement": "unanimous_binary_label" if agree else "disagreement_or_missing",
        "target_review_status": "pending_hayden_review" if agree else "pending_hayden_review_disagreement_or_missing",
        "target_review_source": None,
        "calibration_training_only": cid in calibration_evidence,
        "calibration_evidence_paths": sorted(calibration_evidence.get(cid, [])),
        "calibration_review_status": "recorded_exposure" if cid in calibration_evidence else "prior_exposure_not_fully_audited",
        "source_word_count": src.get("word_count"), "source_text_empty": False,
    }
    for name, label in zip(REVIEWERS, labels):
        row[f"{name}_eligibility"] = reviews[name][cid]["eligibility"]["binary_task"]
        row[f"{name}_target"] = label
        row[f"{name}_review_status"] = reviews[name][cid]["run"]["review_status"]
    metadata.append(row)
require([r["case_id"] for r in metadata] == ids, "Metadata IDs/order mismatch")
require(len(metadata) == len({r["case_id"] for r in metadata}) == 114, "Expected 114 unique rows")
require(all(r["target"] is None and r["case_family_id"] is None for r in metadata), "Unreviewed fields set")
print("Metadata rows:", len(metadata))
print("Provisional target counts:", dict(Counter(r["provisional_target"] for r in metadata)))
print("Target agreement:", dict(Counter(r["target_agreement"] for r in metadata)))
print("Missing separate text-source URLs:", sum(r["text_source_url_missing"] for r in metadata))
print("Reviewed targets: 0; resolved family IDs: 0.")


Metadata rows: 114
Provisional target counts: {0: 75, 1: 39}
Target agreement: {'unanimous_binary_label': 114}
Missing separate text-source URLs: 31
Reviewed targets: 0; resolved family IDs: 0.


## Inspect all 114 cases

This compact view shows every selected case. The JSONL export retains all metadata
columns, including source hashes, source links, and each reviewer's review status.


In [4]:
columns = ["case_id", "case_name", "decision_date", "docket_number", "eligibility",
           "becca_target", "hyunsang_target", "yinkui_target", "provisional_target",
           "target_review_status", "family_review_status", "candidate_related_case_ids",
           "calibration_training_only"]
def format_value(v):
    if v is None:
        return "pending"
    if isinstance(v, list):
        return "; ".join(map(str, v)) or "—"
    return str(v)
header = "<tr>" + "".join(f"<th>{html.escape(c)}</th>" for c in columns) + "</tr>"
body = "".join("<tr>" + "".join(f"<td>{html.escape(format_value(r[c]))}</td>" for c in columns)
               + "</tr>" for r in metadata)
TABLE_HTML = ('<div style="overflow:auto;max-height:650px">'
              '<table style="border-collapse:collapse;font-size:12px" border="1" cellpadding="6">'
              '<caption>114 cases — provisional targets; human review pending</caption>'
              f'<thead>{header}</thead><tbody>{body}</tbody></table></div>')
try:
    from IPython.display import HTML, display
except ImportError:
    print("The complete table is available in the exported metadata_table.html.")
else:
    display(HTML(TABLE_HTML))


case_id,case_name,decision_date,docket_number,eligibility,becca_target,hyunsang_target,yinkui_target,provisional_target,target_review_status,family_review_status,candidate_related_case_ids,calibration_training_only
NY-10090934,Kerr-Inniss v. Williams,2024-08-28,Index No. 710815/21,include,0,0,0,0,pending_hayden_review,not_reviewed,—,False
NY-10128963,"Kavenaght v. 498 Seventh, LLC",2024-10-02,Index No. 518939/18,include,1,1,1,1,pending_hayden_review,not_reviewed,—,False
NY-10160793,Porter v. Stone,2024-10-24,CV-23-1832,include,0,0,0,0,pending_hayden_review,not_reviewed,—,False
NY-10291100,"German v. 333 Rector Garage, LLC",2024-12-10,Ind No. 157393/19 Index No. 157393/19 Appeal No. 3226 Case No. 2023-03457,include,0,0,0,0,pending_hayden_review,not_reviewed,—,False
NY-10309802,Hart v. 210 W. 77 St. LLC,2024-12-31,Index No. 160578/21 Appeal No. 3336 Case No. 2023-06148,include,1,1,1,1,pending_hayden_review,not_reviewed,—,False
NY-2682947,Duluc v. AC & L Food Corp.,2014-07-10,12038 302417/10,include,0,0,0,0,pending_hayden_review,not_reviewed,—,False
NY-2686388,"Benjamin v. Trade Fair Supermarket, Inc.",2014-07-30,2013-11114,include,1,1,1,1,pending_hayden_review,not_reviewed,—,False
NY-2686408,"Mathis v. D.D. Dylan, LLC",2014-07-30,2012-04608,include,0,0,0,0,pending_hayden_review,not_reviewed,—,False
NY-2719389,"Simon v. Hamlet Windwatch Development, LLC",2014-08-20,2013-03542,include,0,0,0,0,pending_hayden_review,not_reviewed,—,False
NY-2757329,Heck v. Regula,2014-12-03,2014-03436,include,1,1,1,1,pending_hayden_review,not_reviewed,—,True


## Export metadata and provenance

Save `metadata.jsonl`, `metadata_table.html`, and `metadata_manifest.json` under
`data/metadata_v1/`. Reruns must reproduce the same version; changed inputs or data
require investigation and a new version. All original inputs remain unchanged.


In [5]:
def json_bytes(v):
    return (json.dumps(v, ensure_ascii=False, indent=2) + "\n").encode("utf-8")
payloads = {
    "metadata.jsonl": "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in metadata).encode("utf-8"),
    "metadata_table.html": ('<!doctype html><html lang="en"><meta charset="utf-8">'
        '<title>MVP case metadata</title><body><h1>MVP metadata: 114 cases</h1>'
        '<p>Provisional labels only. Target and family review pending.</p>'
        + TABLE_HTML + '</body></html>').encode("utf-8"),
}
try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT,
                                     text=True, stderr=subprocess.DEVNULL).strip()
except (OSError, subprocess.CalledProcessError):
    commit = None
manifest = {
    "version": VERSION, "row_count": len(metadata), "selection_version": selection_manifest["version"],
    "repository_commit_at_creation": commit,
    "target_policy": "Unanimous binary labels are provisional; target unset pending Hayden review",
    "family_policy": "Existing hints and Coreano/Sodhi name warnings only; no family adjudication",
    "calibration_policy": "Manifest and saved calibration notes; additional prior exposure requires confirmation",
    "label_meanings": {"0": "fully affirmed", "1": "fully reversed or vacated"},
    "inputs": {n: {"path": str(p.relative_to(ROOT)), "sha256": digest(p)} for n, p in INPUTS.items()},
    "outputs": {n: {"sha256": hashlib.sha256(v).hexdigest()} for n, v in payloads.items()},
}
manifest_path = OUTPUT / "metadata_manifest.json"
if manifest_path.exists():
    previous = read_json(manifest_path)
    require({k:v for k,v in previous.items() if k != "repository_commit_at_creation"}
            == {k:v for k,v in manifest.items() if k != "repository_commit_at_creation"},
            "Metadata inputs/content changed; investigate and choose a new VERSION.")
    manifest = previous
payloads["metadata_manifest.json"] = json_bytes(manifest)
for n, v in payloads.items():
    p = OUTPUT / n
    require(not p.exists() or p.read_bytes() == v, f"Existing {n} differs; choose a new VERSION.")
OUTPUT.mkdir(parents=True, exist_ok=True)
for n, v in payloads.items():
    p = OUTPUT / n
    if not p.exists():
        p.write_bytes(v)
require(read_rows(OUTPUT / "metadata.jsonl") == metadata, "Export verification failed")
print("Saved 114 metadata rows, HTML table, and checksum manifest to", OUTPUT.relative_to(ROOT))


Saved 114 metadata rows, HTML table, and checksum manifest to annotation/submissions/becca/MVP/data/metadata_v1
